# MTH3302 : Méthodes probabilistes et statistiques pour l'I.A.

Jonathan Jalbert<br/>
Professeur agrégé au Département de mathématiques et de génie industriel<br/>
Polytechnique Montréal<br/>


# Chapitre 4 - Analyse en composantes principales


## Contexte : Poids des poissons

On reprend le fichier *fishweights.csv* du TD4 contenant les caractéristiques de 158 poissons péchés dans le lac Laengelmavesi en Finlande. Nous nous intéressons aux poids des poissons (Weight en grammes) en fonction de variables explicatives suivantes :
- l'espèce de poisson (species) ;
- la longueur standard (StandardLength) ;
- la longeur non standard  (NonStandardLength) ;
- la longueur totale (TotalLength) ;
- la hauteur (Height) ;
- la largeur (Width).

On a vu au TD 4 que les variables explicatives étaient fortement multicolinéaire. Nous verrons comment traiter le problème de la multicolinéarité avec l'analyse en composantes principales.

          ___/////___                  _
         /           \    ___          |
       /\             \_ /  /          H
     <   )            __)  \           |
       \/_\\_________/   \__\          _

     |------- SL -------|
     |------- NSL ---------|
     |------- TL ------------|

In [ ]:
using CSV 
using DataFrames
using Distributions
using Gadfly
using LinearAlgebra
using Statistics

In [ ]:
include("functions.jl")

## Chargement des données

On ne s'intéresse qu'à l'espèce "Perche". 

In [ ]:
data = CSV.read("fishweights.csv", DataFrame)
filter!(row -> row.Species == "Perche", data)
first(data,10)

In [ ]:
nrow(data)

## Extraction des matrices

On convertit les variables d'un DataFrame à une matrice.

Pour les poids, on se rappelle qu'au TD4, la relation avec les variables explicatives était linéaire avec la racine cubique des poids. On effectue d'emblée cette transformation ici.

In [ ]:
X = Matrix(data[:, 3:end])

y = (data.Weight).^(1/3)

Z = standardize(X)
standardize!(y)

# 4.1 Rappel d'algèbre linéaire

## 4.1.5 Décomposition en valeurs singulières

In [ ]:
# Décomposition en valeurs singulières de la matrice rectangulaire Z

F = svd(Z)

In [ ]:
# Extraction de la matrice U
U = F.U

In [ ]:
# Extraction de la matrice V
V = F.V

In [ ]:
# Extraction des valeurs singulières
γ = F.S

In [ ]:
γ[1]^2 / sum(γ.^2)

# 4.2 Analyse en composantes principales

In [ ]:
w = round.(V[:,1], digits=2)

println("La première composante principale est composée de : ")
println("\t  T₁ = $(w[1]) Z₁ + $(w[2]) Z₂ + $(w[3]) Z₃ + $(w[4]) Z₄ + $(w[5]) Z₅")

w = round.(V[:,2], digits=2)
println(" ")
println("La deuxième composante principale est composée de : ")
println("\t  T₂ = $(w[1]) Z₁ + $(w[2]) Z₂ + $(w[3]) Z₃ + $(w[4]) Z₄ + $(w[5]) Z₅")

w = round.(V[:,3], digits=2)
println(" ")
println("La troisième composante principale est composée de : ")
println("\t  T₃ = $(w[1]) Z₁ + $(w[2]) Z₂ + $(w[3]) Z₃ + $(w[4]) Z₄ + $(w[5]) Z₅")

w = round.(V[:,4], digits=2)
println(" ")
println("La quatrième composante principale est composée de : ")
println("\t  T₄ = $(w[1]) Z₁ + $(w[2]) Z₂ + $(w[3]) Z₃ + $(w[4]) Z₄ + $(w[5]) Z₅")

w = round.(V[:,5], digits=2)
println(" ")
println("La cinquième composante principale est composée de : ")
println("\t  T₅ = $(w[1]) Z₁ + $(w[2]) Z₂ + $(w[3]) Z₃ + $(w[4]) Z₄ + $(w[5]) Z₅")

## Variance expliquée par les composantes principales

In [ ]:
v = cumsum(γ.^2) / sum(γ.^2)

plot(x=1:5, y=v, Geom.point, Geom.hair,
    Guide.xlabel("Nombre de composantes principales utilisées"), 
    Guide.ylabel("Pourcentage de la variance expliquée"))

Presque 99% de la variance du jeu de donnée est expliquée par la première composante principale. Si on voulait réduire la taille du jeu de données, on pourrait conserver que la première composante principale et disposer des 5 variables explicatives. Seulement 1% de l'information serait perdue pour un jeu de données 5 fois plus petit (une seule variable transformée conservée au lieu des cinq originales). 

# 4.3 Régression sur composantes principales

## Utilisation des composantes les plus importantes

Puisque la première composante explique plus de 98.5% de la variabilité des variables explicatives, on peut utiliser que cette composante comme seule variable explicative. De cette façon, on ne perd presque pas d'information.

In [ ]:
T₁ = Z*V[:,1]

η̂ = T₁\y

plot(x = T₁, y = y, Geom.point, slope=[η̂], Geom.abline(color="red", style=:dash),
    Guide.xlabel("t₁"), Guide.ylabel("∛y"))

## Utilisation de toutes les composantes

In [ ]:
# Calcul de la matrice des composantes principales

T = Z*V

On peut vérifier que les composantes principales ne souffrent pas de multicolinéarité. En effet, toutes les composansantes sont orthogonales.

In [ ]:
compute_VIF(T)

Calcul des coefficients de régression dans l'espace des composantes principales.

In [ ]:
η̂ = T\y

Transformation des coefficients à l'espace des variables explicatives standardisées.

In [ ]:
β̂ = V*η̂

On peut maintenant utiliser les coefficients transformés sur la matrice Z. Les coefficient ont été estimés dans l'espace des composantes principales mais on revient à l'espace orginal pour faciliter les calculs et l'interprétation. 

Ces calculs permettent de trouver les meilleurs coefficients même dans le cas où il y a de la multicolinéarité. Cette approche est très pratique si l'on souhaite utiliser toutes les variables explicatives, même s'il y a multicolinéarité, sans procéder à la fastidieuce tâche de la sélection de modèle.

In [ ]:
ŷ = Z*β̂
e = ŷ - y
plot(x = ŷ, y=e, Geom.point)